# RobustPrompt: Baseline Evaluation Framework
**Goal:** Establish Baseline 1 (Vanilla LLM) and Baseline 2 (LLM + RAG).

### Model
- **** via Groq Inference API (fast, free tier, no CUDA issues)

### Metrics
- **Semantic Stability:** BERTScore F1
- **Factual Accuracy:** Rule-Based Judge

### Fixes Applied
1. Replaced brittle torch/CUDA reinstall with lightweight  SDK — no GPU dependency.
2. HF token read from Colab Secrets () — no  prompt.
3. Model switched to **** (Groq ID: ).
4. All imports consolidated into ONE cell so no NameError on /.
5. BERTScore computed via  directly (avoids  torchvision chain).
6. Plural/possessive keyword matching fixed.
7. AND/OR logic fixed for multi-channel intents.


In [ ]:
!pip install -q bert-score transformers accelerate
print("✅ Dependencies installed.")

✅ Dependencies installed.


In [ ]:
import json, time, re, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from bert_score import score as bert_score_fn
from huggingface_hub import login
from google.colab import userdata

# --- 1. Load your Fine-tuned Llama-3 Model (Generator) ---
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

print("Loading Generator LLM (Llama-3-8B-Instruct)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print(f"✅ Generator LLM loaded successfully on device: {next(model.parameters()).device}")


print("\nLoading Zero-Shot Intent Recognition Model (BART-large-mnli)...")
intent_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0) # Use GPU 0

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Model loaded: meta-llama/Meta-Llama-3-8B-Instruct
   Device: cuda:0


In [ ]:
# ============================================================
# CELL 3 — Load the Knowledge Base (hr_policies.txt)
# ============================================================
INTENT_MAPPING = {
    1: "sick_leave_balance",
    2: "carry_forward_leave",
    3: "maternity_leave_duration",
    4: "paternity_leave_eligibility",
    5: "leave_encashment_policy",
    6: "emergency_leave_company_policy",
    7: "unpaid_leave_policy",
    8: "bereavement_leave",
    9: "medical_certificate_requirement",
    10: "leave_during_notice_period",
}

POLICY_KNOWLEDGE_BASE = {}

with open("hr_policies.txt", "r") as f:
    full_text = f.read()

for block in full_text.split("POLICY")[1:]:
    try:
        policy_id   = int(block.split(":")[0].strip())
        intent_name = INTENT_MAPPING[policy_id]
        content     = ":".join(block.split(":")[1:]).strip()
        POLICY_KNOWLEDGE_BASE[intent_name] = content
    except (ValueError, KeyError) as e:
        print(f"⚠️  Skipped unparseable block: {block[:40]!r}")

print(f"✅ Loaded {len(POLICY_KNOWLEDGE_BASE)} policies into the knowledge base.")
for k, v in POLICY_KNOWLEDGE_BASE.items():
    print(f"  • {k}: {v[:80]}...")

✅ Loaded 10 policies into the knowledge base.
  • sick_leave_balance: SICK LEAVE BALANCE
You can check your remaining sick leave balance by logging in...
  • carry_forward_leave: ANNUAL LEAVE CARRY OVER
You may carry forward a maximum of 10 unused annual leav...
  • maternity_leave_duration: MATERNITY LEAVE
Full-time employees are entitled to 26 weeks of fully paid mater...
  • paternity_leave_eligibility: PATERNITY LEAVE
All full-time employees who have completed 6 months of service a...
  • leave_encashment_policy: LEAVE ENCASHMENT
Leave encashment is permitted only upon resignation or retireme...
  • emergency_leave_company_policy: EMERGENCY LEAVE
For emergency leave, notify your direct manager via phone or Wha...
  • unpaid_leave_policy: UNPAID LEAVE
Unpaid leave of up to 30 days may be granted at the manager's discr...
  • bereavement_leave: BEREAVEMENT LEAVE
Employees are entitled to 5 days of paid bereavement leave for...
  • medical_certificate_requirement: MEDICAL CERTIFICATE


In [ ]:
# ============================================================
# CELL 4 — Rule-Based Judge (FIXED)
# ============================================================

def _match(answer_lower: str, keyword: str) -> bool:
    """Flexible keyword matcher — handles plurals and possessives."""
    stem = re.sub(r"'s?$", "", keyword.lower())   # manager's -> manager
    stem = re.sub(r"s$",   "", stem)              # days -> day
    return bool(re.search(re.escape(stem), answer_lower))


KEY_FACTS = {
    "sick_leave_balance": {
        "all": ["hr portal", "january 1st"],
    },
    "carry_forward_leave": {
        "all": ["10", "december 31st"],
    },
    "maternity_leave_duration": {
        "all": ["26 week"],
    },
    "paternity_leave_eligibility": {
        "all": ["10 day", "6 month"],
    },
    "leave_encashment_policy": {
        "all": ["30 day"],
        "any": [["resignation"], ["retirement"]],
    },
    "emergency_leave_company_policy": {
        "all": ["manager", "24 hour"],
        "any": [["phone"], ["whatsapp"], ["email"], ["message"]],
    },
    "unpaid_leave_policy": {
        "all": ["30 day", "manager", "discretion"],
    },
    "bereavement_leave": {
        "all": ["immediate"],
        "any": [["5 day"], ["5-day"]],
    },
    "medical_certificate_requirement": {
        "all": ["2"],
        "any": [["consecutive day"], ["48 hour"], ["two day"], ["2 day"]],
    },
    "leave_during_notice_period": {
        "all": ["approval"],
        "any": [["manager"], ["hr"]],
    },
}


def is_factually_correct_rule_based(generated_answer: str, intent_name: str) -> bool:
    if intent_name not in KEY_FACTS:
        return False
    rules        = KEY_FACTS[intent_name]
    answer_lower = generated_answer.lower()
    for kw in rules.get("all", []):
        if not _match(answer_lower, kw):
            return False
    any_groups = rules.get("any", [])
    if any_groups:
        if not any(all(_match(answer_lower, kw) for kw in g) for g in any_groups):
            return False
    return True


# Sanity checks
assert _match("the employee gets 10 days carry forward", "10 day"),  "plural fail"
assert _match("26 weeks of maternity leave",             "26 week"),  "plural fail"
assert _match("at the manager's discretion",             "manager's discretion"), "possessive fail"
assert _match("notify manager within 24 hours via phone","24 hour"),  "plural fail"
print("✅ All _match() sanity checks passed.")

✅ All _match() sanity checks passed.


In [ ]:
# ============================================================
# CELL 5 — Helper: LLM Inference Functions
# ============================================================
VANILLA_SYS_PROMPT = (
    "You are a helpful HR assistant. "
    "Answer the employee's question accurately and concisely."
)

RAG_SYS_PROMPT = (
    "You are an HR Assistant. "
    "Use ONLY the provided 'POLICY CONTEXT' to answer. "
    "Be direct and fact-based. "
    "Include the specific numbers and dates from the policy verbatim."
)


def _call_api(messages: list, max_tokens: int = 200) -> str:
    try:
        input_ids = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            tokenize=True,
        )
        if hasattr(input_ids, "input_ids"):
            input_ids = input_ids.input_ids
        input_ids = input_ids.to(model.device)

        # Create attention mask explicitly to suppress the warning
        attention_mask = torch.ones_like(input_ids)

        with torch.no_grad():
            output = model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_tokens,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        new_tokens = output[0][input_ids.shape[-1]:]
        return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    except Exception as e:
        import traceback
        traceback.print_exc()
        return ""

def get_vanilla_response(prompt_text: str) -> str:
    """Baseline 1 — no context, pure LLM."""
    return _call_api([
        {"role": "system", "content": VANILLA_SYS_PROMPT},
        {"role": "user",   "content": prompt_text},
    ])


def get_rag_response(prompt_text: str, policy_context: str) -> str:
    """Baseline 2 — RAG: full policy injected as context."""
    user_msg = "POLICY CONTEXT:\n---\n" + policy_context + "\n---\n\nUSER QUESTION: " + prompt_text
    return _call_api([
        {"role": "system", "content": RAG_SYS_PROMPT},
        {"role": "user",   "content": user_msg},
    ])


print("✅ API helper functions defined.")

✅ API helper functions defined.


In [ ]:
def run_evaluation(eval_clusters, get_response_fn, label: str) -> dict:
    results                  = []
    overall_stability_scores = []
    total_answers            = 0
    correct_answers          = 0

    sep = "=" * 60
    print(sep)
    print("  Starting: " + label)
    print(sep)

    for cluster in eval_clusters:
        intent_name      = cluster["intent"]
        canonical_prompt = cluster["canonical_prompt"]

        print("\n▶ Intent: " + intent_name)

        # Canonical answer
        canonical_answer = get_response_fn(canonical_prompt)
        time.sleep(0.5)

        canon_pass = is_factually_correct_rule_based(canonical_answer, intent_name)
        total_answers   += 1
        correct_answers += int(canon_pass)
        marker = "✅" if canon_pass else "❌"
        print("  " + marker + " Canonical: " + canonical_answer[:120])

        # Paraphrase answers
        paraphrase_answers = []
        for p in cluster["paraphrases"]:
            ans = get_response_fn(p)
            time.sleep(0.5)
            para_pass = is_factually_correct_rule_based(ans, intent_name)
            total_answers   += 1
            correct_answers += int(para_pass)
            paraphrase_answers.append(ans)
            marker = "✅" if para_pass else "❌"
            print("  " + marker + " Paraphrase: " + ans[:120])

        # BERTScore
        refs = [canonical_answer] * len(paraphrase_answers)
        P, R, F1 = bert_score_fn(
            paraphrase_answers, refs,
            model_type="roberta-large",
            verbose=False,
        )
        avg_stability = F1.mean().item()
        overall_stability_scores.append(avg_stability)
        print("  📊 Cluster BERTScore F1: " + str(round(avg_stability, 4)))

        results.append({
            "intent":               intent_name,
            "canonical_prompt":     canonical_prompt,
            "canonical_answer":     canonical_answer,
            "paraphrases":          cluster["paraphrases"],
            "paraphrase_answers":   paraphrase_answers,
            "cluster_stability_f1": avg_stability,
        })

    final_stability = sum(overall_stability_scores) / len(overall_stability_scores)
    final_accuracy  = (correct_answers / total_answers) * 100

    print(sep)
    print("  SCORECARD — " + label)
    print("  Semantic Stability  (BERTScore F1) : " + str(round(final_stability, 4)))
    print("  Factual Accuracy    (Rule-Based)   : " + str(round(final_accuracy, 2)) + "%  (" + str(correct_answers) + "/" + str(total_answers) + " correct)")
    print(sep)

    return {
        "results":               results,
        "final_stability_score": final_stability,
        "final_accuracy_score":  final_accuracy,
    }

print("✅ Evaluation loop helper defined.")

✅ Evaluation loop helper defined.


In [ ]:
# ============================================================
# CELL 7 — Load Evaluation Data
# ============================================================
with open("hr_evaluation_clusters.json", "r") as f:
    eval_clusters = json.load(f)

print(f"✅ Loaded {len(eval_clusters)} evaluation clusters.")
for c in eval_clusters:
    print(f"  • {c['intent']}  ({1 + len(c['paraphrases'])} prompts)")

✅ Loaded 10 evaluation clusters.
  • sick_leave_balance  (5 prompts)
  • carry_forward_leave  (5 prompts)
  • maternity_leave_duration  (5 prompts)
  • paternity_leave_eligibility  (5 prompts)
  • leave_encashment_policy  (5 prompts)
  • emergency_leave_company_policy  (5 prompts)
  • unpaid_leave_policy  (5 prompts)
  • bereavement_leave  (5 prompts)
  • medical_certificate_requirement  (5 prompts)
  • leave_during_notice_period  (5 prompts)


In [ ]:
# DEBUG — run this before Cell 8 to see the real error
test_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Say hello."},
]

try:
    input_ids = tokenizer.apply_chat_template(
        test_messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=50,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0][input_ids.shape[-1]:]
    print("✅ Output: " + tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

except Exception as e:
    import traceback
    traceback.print_exc()  # prints the FULL error

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py", line 275, in __getattr__
    return self.data[item]
           ~~~~~~~~~^^^^^^
KeyError: 'shape'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/tmp/ipykernel_14036/2613025568.py", line 15, in <cell line: 0>
    output = model.generate(
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py", line 2535, in generate
    batch_size = inputs_tensor.shape[0]
                 ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py", line 277, in __getattr__
    raise AttributeError
AttributeError


In [ ]:
# ============================================================
# CELL 8 — Run Baseline 1 (Vanilla LLM — llama-3-8b-instant)
# ============================================================
baseline1_output = run_evaluation(
    eval_clusters=eval_clusters,
    get_response_fn=get_vanilla_response,
    label=f"Baseline 1 — Vanilla LLM ({MODEL_ID})",
)

with open("baseline_1_results.json", "w") as f:
    json.dump(baseline1_output["results"], f, indent=2)

print(" Detailed results saved → baseline_1_results.json")

  Starting: Baseline 1 — Vanilla LLM (meta-llama/Meta-Llama-3-8B-Instruct)

▶ Intent: sick_leave_balance
  ❌ Canonical: According to our company's records, you have used 5 sick leave days so far this year. You are entitled to a total of 12 
  ❌ Paraphrase: I'd be happy to help you with that. Let me check our system real quick. *checks system* Okay, it looks like you have 5 s
  ❌ Paraphrase: I'd be happy to help you with that. As of our current system, you have 10 days of sick leave available for the current y
  ❌ Paraphrase: Don't worry, I'm happy to help clarify! To check your remaining sick leave balance, you can follow these steps:

1. Log 
  ❌ Paraphrase: Yes, you can check your remaining sick leave balance online through our company's employee portal. To do so, follow thes


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  📊 Cluster BERTScore F1: 0.8684

▶ Intent: carry_forward_leave
  ❌ Canonical: According to our company's policy, unused annual leave can be carried forward to the next year, but there are some limit
  ❌ Paraphrase: According to our company's policy, unused vacation days can be carried over to the next year, but there are some limitat
  ✅ Paraphrase: According to our company's vacation policy, you can carry over up to 10 unused vacation days to the next calendar year. 
  ❌ Paraphrase: According to our company's vacation policy, employees are allowed to carry over up to 10 days of unused vacation time to
  ❌ Paraphrase: According to our company's policies, you can carry over up to 40 hours of unused vacation time from this year to next ye


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  📊 Cluster BERTScore F1: 0.8959

▶ Intent: leave_during_notice_period
  ✅ Canonical: According to our company's policy, annual leave can be taken during the notice period, but it's subject to certain condi
  ✅ Paraphrase: As per our company's policy, annual leave can be taken during the notice period, but it's subject to approval from your 
  ❌ Paraphrase: According to our company's policies, employees are entitled to take annual leave during their notice period, but there a
  ✅ Paraphrase: According to our company's policy, employees are not allowed to take holidays during their notice period. This is to ens
  ✅ Paraphrase: I understand your concern! In cases where an employee's annual leave coincides with their notice period, the company's p


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  📊 Cluster BERTScore F1: 0.8994
  SCORECARD — Baseline 1 — Vanilla LLM (meta-llama/Meta-Llama-3-8B-Instruct)
  Semantic Stability  (BERTScore F1) : 0.8891
  Factual Accuracy    (Rule-Based)   : 10.0%  (5/50 correct)
 Detailed results saved → baseline_1_results.json


In [ ]:
# ============================================================
# CELL 9 — Run Evaluation 2 (LLM + RAG with ZERO-SHOT INTENT RECOGNITION)
# ============================================================
# Get the list of all possible intents for the classifier
intent_labels = list(POLICY_KNOWLEDGE_BASE.keys())

def rag_fn(prompt_text: str) -> str:

    # Step 1: DYNAMIC INTENT RECOGNITION
    # The zero-shot classifier predicts the intent from the raw text
    classification_result = intent_classifier(prompt_text, intent_labels)
    predicted_intent = classification_result['labels'][0]


    # Step 2: DETERMINISTIC RETRIEVAL
    # Use the predicted intent to look up the policy in our knowledge base
    context = POLICY_KNOWLEDGE_BASE.get(predicted_intent, "Policy not found.")

    # Step 3: AUGMENT & GENERATE
    return get_rag_response(prompt_text, context)


# --- Run the full evaluation with the new RAG pipeline ---
baseline2_output = run_evaluation(
    eval_clusters=eval_clusters,
    get_response_fn=rag_fn,
    label="RobustPrompt Pipeline (Zero-Shot Intent Recognition + rDPO Generation)",
)

with open("/kaggle/working/adapter_rag_results.json", "w") as f:
    json.dump(baseline2_output["results"], f, indent=2)
print("✅ Detailed results saved → /kaggle/working/adapter_rag_results.json")

  Starting: Baseline 2 — LLM + RAG (full policy context injected)

▶ Intent: sick_leave_balance
  ❌ Canonical: According to the policy, you can check your remaining sick leave balance by logging into the HR portal under My Leave > 
  ❌ Paraphrase: According to the policy, you can check your remaining sick leave balance by logging into the HR portal under My Leave > 
  ❌ Paraphrase: According to the policy, you can check your remaining sick leave balance by logging into the HR portal under My Leave > 
  ❌ Paraphrase: According to the policy, you can check your remaining sick leave balance by logging into the HR portal under My Leave > 
  ❌ Paraphrase: According to the policy, you can check your remaining sick leave balance by logging into the HR portal under My Leave > 


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  📊 Cluster BERTScore F1: 1.0

▶ Intent: carry_forward_leave
  ❌ Canonical: According to the policy, yes, you can carry forward unused annual leave to the next year, but with a maximum limit of 10
  ✅ Paraphrase: According to our policy, you can carry forward a maximum of 10 unused annual leave days to the next calendar year. Since
  ✅ Paraphrase: According to our policy, you can carry forward a maximum of 10 unused annual leave days to the next calendar year. If yo
  ❌ Paraphrase: According to the policy, you can carry forward a maximum of 10 unused annual leave days to the next calendar year.
  ❌ Paraphrase: According to our policy, you may carry forward a maximum of 10 unused annual leave days to the next calendar year.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  📊 Cluster BERTScore F1: 0.9507

▶ Intent: leave_during_notice_period
  ✅ Canonical: According to the policy, annual leave during the notice period requires written approval from your manager and HR. Howev
  ✅ Paraphrase: According to the policy, annual leave during the notice period requires written approval from your manager and HR. There
  ✅ Paraphrase: According to the policy, annual leave during the notice period requires written approval from your manager and HR. The c
  ✅ Paraphrase: According to the policy, annual leave during the notice period requires written approval from your manager and HR. The c
  ✅ Paraphrase: According to the policy, annual leave during the notice period requires written approval from your manager and HR. Since


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  📊 Cluster BERTScore F1: 0.946
  SCORECARD — Baseline 2 — LLM + RAG (full policy context injected)
  Semantic Stability  (BERTScore F1) : 0.9447
  Factual Accuracy    (Rule-Based)   : 72.0%  (36/50 correct)
 Detailed results saved → baseline_2_rag_results.json


In [ ]:
sep = "=" * 60
print(sep)
print("  FINAL COMPARISON TABLE")
print(sep)
print("  " + "Metric".ljust(35) + "B1 Vanilla".rjust(12) + "B2 RAG".rjust(12))
print("  " + "-"*35 + " " + "-"*12 + " " + "-"*12)
print("  " + "Semantic Stability (BERTScore F1)".ljust(35) +
      str(round(baseline1_output['final_stability_score'], 4)).rjust(12) +
      str(round(baseline2_output['final_stability_score'], 4)).rjust(12))
print("  " + "Factual Accuracy (Rule-Based %)".ljust(35) +
      (str(round(baseline1_output['final_accuracy_score'], 2)) + "%").rjust(12) +
      (str(round(baseline2_output['final_accuracy_score'], 2)) + "%").rjust(12))
print(sep)
print()
print("Expected outcome after fixes:")
print("  B1 Vanilla  → low factual accuracy (hallucination, no context)")
print("  B2 RAG      → substantially higher factual accuracy")
print("  Both        → high semantic stability (B2 should be even higher)")

  FINAL COMPARISON TABLE
  Metric                               B1 Vanilla      B2 RAG
  ----------------------------------- ------------ ------------
  Semantic Stability (BERTScore F1)        0.8891      0.9447
  Factual Accuracy (Rule-Based %)           10.0%       72.0%

Expected outcome after fixes:
  B1 Vanilla  → low factual accuracy (hallucination, no context)
  B2 RAG      → substantially higher factual accuracy
  Both        → high semantic stability (B2 should be even higher)
